In [1]:
import numpy as np 
import healpy as hp 
#import matplotlib.pyplot as plt
from astropy.time import Time

from fftvis import simulate_vis

import matvis
#from hera_sim.antpos import hex_array
#from hera_cal import redcal
from pyuvdata import GaussianBeam
from pyuvdata.telescopes import Telescope

from pyuvdata.analytic_beam import AiryBeam

# plt.rcParams['font.family'] = 'serif'
# plt.rcParams['mathtext.fontset'] = 'dejavuserif'

In [2]:
# Reproducible random state
rng = np.random.default_rng(42)

# Antenna positions — hex array of 19 antennas
#antpos = hex_array(4, split_core=True, outriggers=0)
#nants = len(antpos)

# Observation frequency and times
freq = 150e6                              # single frequency (Hz)
#times = np.linspace(2459959, 2459959.5, 4)
telescope_loc = Telescope.from_known_telescopes("hera").location

# Sky model — HEALPix map of random point sources
nside = 64
npix = hp.nside2npix(nside)
dec, ra = hp.pix2ang(nside, np.arange(npix))
dec -= np.pi / 2
flux = np.abs(rng.standard_normal((npix, 1)))  # (nsrc, nfreq)

In [3]:
# g = GaussianBeam(diameter=14.0)
beam = AiryBeam(diameter=14.0)

In [4]:
a = np.load('/Users/user/Downloads/dspec.npz')
dspec = a['dspec']
tlim = a['tlim']
print(tlim)
print(dspec[:,0])

3600.0000000000005
[0.51046544 1.1614939  3.5047772  0.46863437 0.7753495  1.003375
 0.07837769 1.018013  ]


In [5]:
t = Time('2010-01-01T04:30:00', format='isot', scale='utc')
t.jd

np.float64(2455197.6875)

In [ ]:
# beamfile = '/Users/user/Downloads/NF_HERA_Vivaldi_efield_beam.fits'

# from pyuvdata import UVBeam

# beam = UVBeam.from_file(beamfile)

# beam.check()

In [6]:
antpos = {0: np.array([-29.2       ,  42.14656965,   0.        ]), 1: np.array([-14.6       ,  42.14656965,   0.        ])}
print(antpos)

{0: array([-29.2       ,  42.14656965,   0.        ]), 1: array([-14.6       ,  42.14656965,   0.        ])}


In [7]:
dspec.shape
print(dspec[:,0])

[0.51046544 1.1614939  3.5047772  0.46863437 0.7753495  1.003375
 0.07837769 1.018013  ]


In [8]:
# Simulate visibilities with the CPU backend (default)
print('Starting simulate_vis')
vis = simulate_vis(
    ants=antpos,
    fluxes=flux,
    ra=ra,
    dec=dec,
    freqs=np.array([freq]),
    # fluxes=np.array([dspec[:,0]]).astype('float64'),
    # ra=np.array([6.5/24 * 2*np.pi]),
    # dec=np.array([-(28+34/60)*np.pi/180]),
    # freqs=np.linspace(150e6, 151e6, num=len(dspec[:,0])),
    times=np.array([t.jd]),
    beam=beam,
    polarized=False,
    backend='cpu',
    # precision=2,
    # trace_mem=False, 
    # nprocesses=2,
    telescope_loc=telescope_loc,
    beam_spline_opts={"order": 1},
    coord_method_params={"update_bcrs_every": 1e9},
    coord_method="CoordinateRotationERFA",
    #baselines=[(0,1)]
)

Starting simulate_vis


: 